# 00 — Load and sanity

`dmercator3d` is independent of `notebooks/dmercator/`. Default run: **d3**.


**Paths** — set `RUN_SUBDIR` or env `DMERCATOR_RUN`.


### Run paths and file checks

**Method:** `paths_for_run` builds absolute paths to `edges_GC.inf_coord`, `.edge`, and `.inf_log` under `d-mercator-run/<subdir>/`. `DMERCATOR_RUN` (or `RUN_SUBDIR` in the notebook) selects which run folder to use.

**How to read the output:** Each line should show `exists=True` once you have produced a D-Mercator run. If any path is missing, fix the subdir name or generate that run before downstream notebooks.


In [1]:
import os
from pathlib import Path

import dmercator3d_io as dm

RUN_SUBDIR = "d3"
RUN_SUBDIR = os.environ.get("DMERCATOR_RUN", RUN_SUBDIR)
RUN_DIR = dm.get_run_dir(RUN_SUBDIR)
paths = dm.paths_for_run(RUN_SUBDIR)
print("RUN_DIR:", RUN_DIR.resolve())
for k, v in paths.items():
    print(f"  {k}: {v} exists={v.is_file()}")


RUN_DIR: C:\Users\john\Documents\philipp\hbol\d-mercator-run\d3
  inf_coord: C:\Users\john\Documents\philipp\hbol\d-mercator-run\d3\edges_GC.inf_coord exists=True
  edge: C:\Users\john\Documents\philipp\hbol\d-mercator-run\d3\edges_GC.edge exists=True
  inf_log: C:\Users\john\Documents\philipp\hbol\d-mercator-run\d3\edges_GC.inf_log exists=False


**Load** — `parse_inf_coord` detects `Inf.Pos.*` count from the header; merge graph degree.


### Parse coordinates, attach degrees, write cache

**Method:** `parse_inf_coord` reads the vertex table: variable count of `Inf.Pos.*` from the header, then each row’s κ, hyperbolic radius, direction components, and vertex label (right-aligned numeric tail). `load_edges_graph` reads the PPI edgelist. Degrees from `G` are joined onto the table; `merged.parquet` avoids re-parsing in later notebooks.

**How to read the output:** `n_pos` should be 4 for `DIMENSION 3`. `coord rows` should match `nb_vertices` when the header reports it. `|V|` / `|E|` describe the loaded graph (vertices only in the edge file appear in `G`). `merged.head()` previews columns including `degree` (0 if a coordinate row has no edges).


In [2]:
meta, df = dm.parse_inf_coord(paths["inf_coord"])
G = dm.load_edges_graph(paths["edge"])
print("meta keys:", sorted(meta.keys()))
print("n_pos:", meta.get("n_pos"), "dimension (footer):", meta.get("dimension"))
print("coord rows:", len(df), "graph |V|:", G.number_of_nodes(), "|E|:", G.number_of_edges())
nbv = meta.get("nb_vertices")
if nbv is None:
    print("nb_vertices not found in header comments")
elif int(nbv) != len(df):
    print("WARN: coord rows", len(df), "!= meta nb_vertices", nbv)
else:
    print("coord rows matches nb_vertices:", len(df))

deg = dict(G.degree())
merged = df.assign(degree=df["Vertex"].map(deg))
merged["degree"] = merged["degree"].fillna(0).astype(int)

cache = Path("cache")
cache.mkdir(exist_ok=True)
out_p = cache / "merged.parquet"
dm.save_merged_parquet(merged, out_p)
print("wrote", out_p.resolve())
merged.head()


meta keys: ['beta', 'dimension', 'inf_coord_path', 'kappa_min', 'mu', 'n_pos', 'nb_vertices', 'radius_h_d1', 'radius_s_d']
n_pos: 4 dimension (footer): 3
coord rows: 17090 graph |V|: 17090 |E|: 298866
coord rows matches nb_vertices: 17090
wrote C:\Users\john\Documents\philipp\hbol\notebooks\dmercator3d\cache\merged.parquet


,Vertex,Inf.Kappa,Inf.Hyp.Rad,Inf.Pos.1,Inf.Pos.2,Inf.Pos.3,Inf.Pos.4,degree
0,AR,1765.8100,8.26924,2.41513,3.969410,4.73157,-6.845640,258
1,C2,40.3874,10.78780,3.53395,-1.583980,-5.22754,6.965240,12
2,C3,342.6260,9.36239,-7.26750,2.158880,4.18761,3.978220,62
3,C5,113.2300,10.10050,4.47360,0.008817,-2.70805,7.968240,25
4,C6,25.4012,11.09700,-9.31054,-0.216400,1.99079,0.378622,5
